# Contrastive Evidence Fusion — MVP

## Architecture (two-stage)

### Stage 1: Contrastive Pretraining (label-free)
Learn a spectral encoder from **27K library spectra** using InChIKey-14 as free supervision.
Same compound (same IK14) = positive pair, different compound = negative pair.
The encoder learns "what makes two spectra chemically equivalent" without any human curation labels.

- Input: binned MS/MS spectrum (500-dim, 1 Da bins over 0–500 m/z)
- Encoder: MLP → 64-dim embedding
- Loss: InfoNCE (contrastive) — pull same-compound spectra together, push different-compound apart
- Data: 27K library spectra, 40K positive pairs, 10K unique compounds

### Stage 2: Evidence Fusion (fine-tuned)
For each spectrum, compute the **reference embedding** from the pretrained encoder,
then fuse it with scalar evidence channels → P(correct top-1).

```
ref_spectrum → [pretrained encoder] → ref_embed (64-dim)
                                          ↓
                           concat with scalar features (12-dim)
                                          ↓
                              [Fusion MLP] → P(correct)
```

The encoder is fine-tuned end-to-end — the spectral representation adapts to the confidence task.

### Why this matters
- Stage 1 uses **chemistry** (IK14), not Oliver's labels — sidesteps label noise
- The encoder learns spectral patterns (fragment importance, neutral losses) that entropy_similarity treats uniformly
- The fusion head learns interactions between spectral quality and metadata (RT, adducts)
- The spectral embedding is a **new feature** that captures information beyond the 12 hand-crafted scalars

In [ ]:
import sys, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

# ── Load everything ──
ft = pd.read_csv('../data/feature_table_v2.csv')
with open('../data/library_peaks_cache.json') as f:
    lib_peaks_cache = json.load(f)

top1 = (ft.sort_values('entropy_similarity', ascending=False)
          .groupby('wiki_id').first().reset_index())
labels = top1['hit_label'].values

print(f'Feature table: {len(ft):,} rows')
print(f'Library peaks cache: {len(lib_peaks_cache):,} spectra')
print(f'Top-1 spectra: {len(top1):,} (TP={labels.sum():,}, FP={(labels==0).sum():,})')

## Step 1 — Spectrum binning

Convert variable-length peak lists into fixed-size vectors for the encoder.
Each spectrum becomes a 500-dim vector: m/z bins from 0 to 500 Da at 1 Da resolution.
Intensities are normalized to sum=1 (probability distribution over fragment bins).

In [ ]:
N_BINS = 500  # 0-500 m/z at 1 Da resolution
EMBED_DIM = 64

def bin_spectrum(peaks, n_bins=N_BINS):
    """Convert [(mz, intensity), ...] → fixed-size binned vector, normalized to sum=1."""
    vec = np.zeros(n_bins, dtype=np.float32)
    if not peaks:
        return vec
    for mz, intensity in peaks:
        b = int(mz)
        if 0 <= b < n_bins:
            vec[b] += intensity
    total = vec.sum()
    if total > 0:
        vec /= total
    return vec

# ── Bin all library spectra that have IK14 (for contrastive pretraining) ──
lib_ik14_map = ft[['library_wiki_id', 'hit_ik14']].dropna().drop_duplicates()
lib_ik14_map = lib_ik14_map[lib_ik14_map['hit_ik14'] != '']
lib_ik14_dict = dict(zip(lib_ik14_map['library_wiki_id'], lib_ik14_map['hit_ik14']))

# Only keep spectra with 2+ per IK14 (needed for contrastive pairs)
ik14_to_libs = defaultdict(list)
for lib_id, ik14 in lib_ik14_dict.items():
    if lib_id in lib_peaks_cache:
        ik14_to_libs[ik14].append(lib_id)

# Filter to IK14s with 2+ spectra
ik14_to_libs = {k: v for k, v in ik14_to_libs.items() if len(v) >= 2}

# Build binned matrix and IK14 labels
lib_ids_for_pretrain = []
lib_ik14_labels = []
for ik14, lib_ids in ik14_to_libs.items():
    for lid in lib_ids:
        lib_ids_for_pretrain.append(lid)
        lib_ik14_labels.append(ik14)

print(f'Pretraining set: {len(lib_ids_for_pretrain):,} spectra, '
      f'{len(ik14_to_libs):,} compounds (IK14s with 2+ spectra)')

# Bin them all
t0 = time.time()
pretrain_spectra = np.stack([bin_spectrum(lib_peaks_cache[lid]) for lid in lib_ids_for_pretrain])
print(f'Binned {len(pretrain_spectra):,} spectra in {time.time()-t0:.1f}s')
print(f'Binned matrix shape: {pretrain_spectra.shape}')
print(f'Non-zero bins per spectrum: median={np.median((pretrain_spectra > 0).sum(axis=1)):.0f}')

# ── Also bin reference spectra for top-1 (for Stage 2) ──
ref_binned = np.zeros((len(top1), N_BINS), dtype=np.float32)
ref_available = np.zeros(len(top1), dtype=bool)
for i, lid in enumerate(top1['library_wiki_id']):
    if lid in lib_peaks_cache:
        ref_binned[i] = bin_spectrum(lib_peaks_cache[lid])
        ref_available[i] = True

print(f'\nTop-1 reference spectra binned: {ref_available.sum():,}/{len(top1):,} '
      f'({100*ref_available.mean():.1f}%)')

## Step 2 — Spectral encoder + contrastive pretraining

The encoder maps a 500-dim binned spectrum → 64-dim embedding.

**InfoNCE loss**: for each anchor spectrum, we have one positive (same compound) and
many negatives (different compounds) in the batch. The loss maximizes the similarity
to the positive relative to all negatives:

```
L = -log( exp(sim(anchor, positive) / tau) / sum_j exp(sim(anchor, neg_j) / tau) )
```

This is the same loss used in CLIP, SimCLR, etc. Temperature `tau` controls sharpness.

In [ ]:
class SpectralEncoder(nn.Module):
    """Encodes a binned MS/MS spectrum into a fixed-size embedding."""

    def __init__(self, input_dim=N_BINS, embed_dim=EMBED_DIM):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, embed_dim),
        )

    def forward(self, x):
        """Returns L2-normalized embeddings."""
        h = self.encoder(x)
        return F.normalize(h, p=2, dim=-1)


class ContrastivePairDataset(Dataset):
    """Yields (anchor, positive) pairs from spectra grouped by IK14."""

    def __init__(self, spectra_matrix, ik14_labels):
        self.spectra = spectra_matrix
        # Group indices by IK14
        self.ik14_to_idx = defaultdict(list)
        for i, ik in enumerate(ik14_labels):
            self.ik14_to_idx[ik].append(i)
        # Only keep groups with 2+ spectra
        self.valid_ik14s = [k for k, v in self.ik14_to_idx.items() if len(v) >= 2]
        # Build flat list of (anchor_idx, ik14) for sampling
        self.anchors = []
        for ik in self.valid_ik14s:
            for idx in self.ik14_to_idx[ik]:
                self.anchors.append((idx, ik))

    def __len__(self):
        return len(self.anchors)

    def __getitem__(self, i):
        anchor_idx, ik14 = self.anchors[i]
        # Sample a different spectrum of the same compound as positive
        group = self.ik14_to_idx[ik14]
        pos_idx = anchor_idx
        while pos_idx == anchor_idx:
            pos_idx = group[np.random.randint(len(group))]
        return (torch.tensor(self.spectra[anchor_idx]),
                torch.tensor(self.spectra[pos_idx]))


def info_nce_loss(anchors, positives, temperature=0.07):
    """
    InfoNCE contrastive loss.
    anchors, positives: [batch, embed_dim], L2-normalized.
    Negatives are all other samples in the batch.
    """
    batch_size = anchors.shape[0]
    # Similarity matrix: [batch, batch]
    sim = torch.mm(anchors, positives.t()) / temperature
    # Positive pairs are on the diagonal
    targets = torch.arange(batch_size, device=sim.device)
    return F.cross_entropy(sim, targets)


def pretrain_encoder(encoder, dataset, epochs=30, batch_size=512, lr=1e-3):
    """Contrastive pretraining loop."""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        drop_last=True, num_workers=0)
    optimizer = optim.Adam(encoder.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = []
    for epoch in range(epochs):
        encoder.train()
        epoch_loss = 0
        n_batches = 0
        for anchor_batch, pos_batch in loader:
            anchor_emb = encoder(anchor_batch)
            pos_emb = encoder(pos_batch)
            loss = info_nce_loss(anchor_emb, pos_emb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        scheduler.step()
        avg_loss = epoch_loss / max(n_batches, 1)
        history.append(avg_loss)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1:3d}: loss={avg_loss:.4f}')

    return history


print(f'SpectralEncoder: {sum(p.numel() for p in SpectralEncoder().parameters()):,} params')
print(f'Contrastive pairs dataset: {len(ContrastivePairDataset(pretrain_spectra, lib_ik14_labels)):,} anchors')

## Step 3 — Run contrastive pretraining

This learns the spectral representation. No Oliver labels are used — only IK14.

In [ ]:
torch.manual_seed(42)
encoder = SpectralEncoder()
dataset = ContrastivePairDataset(pretrain_spectra, lib_ik14_labels)

print(f'Contrastive pretraining on {len(dataset):,} anchors, {len(ik14_to_libs):,} compounds...')
t0 = time.time()
loss_history = pretrain_encoder(encoder, dataset, epochs=30, batch_size=512, lr=1e-3)
print(f'Pretraining done in {time.time()-t0:.1f}s')

# Plot loss curve
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(loss_history, 'darkorange', lw=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('InfoNCE loss')
ax.set_title('Contrastive pretraining loss')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## Step 4 — Visualize the learned embedding space

Do same-compound spectra actually cluster together? UMAP/t-SNE of the learned embeddings,
colored by compound identity.

In [ ]:
# Compute embeddings for a random subset (for visualization)
encoder.eval()
n_viz = min(5000, len(pretrain_spectra))
viz_idx = np.random.choice(len(pretrain_spectra), n_viz, replace=False)

with torch.no_grad():
    viz_emb = encoder(torch.tensor(pretrain_spectra[viz_idx])).numpy()

viz_ik14 = [lib_ik14_labels[i] for i in viz_idx]

# Color by compound: top 20 most frequent compounds get unique colors, rest gray
from collections import Counter
ik14_counts = Counter(viz_ik14)
top20 = [ik for ik, _ in ik14_counts.most_common(20)]

try:
    from sklearn.manifold import TSNE
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    coords = tsne.fit_transform(viz_emb)

    fig, ax = plt.subplots(figsize=(10, 8))

    # Gray background for uncommon compounds
    other_mask = np.array([ik not in top20 for ik in viz_ik14])
    ax.scatter(coords[other_mask, 0], coords[other_mask, 1],
               s=3, alpha=0.1, color='lightgray', label='other')

    # Colored points for top 20 compounds
    cmap = plt.cm.tab20
    for j, ik in enumerate(top20):
        mask = np.array([v == ik for v in viz_ik14])
        n = mask.sum()
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   s=15, alpha=0.7, color=cmap(j / 20),
                   label=f'{ik[:8]}.. (n={n})')

    ax.set_title('t-SNE of learned spectral embeddings (colored by compound)')
    ax.legend(fontsize=6, ncol=2, loc='upper right', markerscale=2)
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

    # Quantitative check: mean intra-compound distance vs inter-compound distance
    intra_dists = []
    inter_dists = []
    for ik in top20:
        mask = np.array([v == ik for v in viz_ik14])
        emb_group = viz_emb[mask]
        if len(emb_group) >= 2:
            # Intra: pairwise distances within group
            for a in range(len(emb_group)):
                for b in range(a+1, min(a+5, len(emb_group))):
                    intra_dists.append(np.linalg.norm(emb_group[a] - emb_group[b]))
        # Inter: distance to random other embeddings
        other_emb = viz_emb[~mask]
        for a in range(min(10, len(emb_group))):
            ri = np.random.randint(len(other_emb))
            inter_dists.append(np.linalg.norm(emb_group[a] - other_emb[ri]))

    print(f'Intra-compound distance: {np.mean(intra_dists):.3f} (same compound spectra)')
    print(f'Inter-compound distance: {np.mean(inter_dists):.3f} (different compound spectra)')
    print(f'Ratio (lower=better clustering): {np.mean(intra_dists)/np.mean(inter_dists):.3f}')

except ImportError:
    print('sklearn.manifold.TSNE not available — skipping visualization')

## Step 5 — Evidence Fusion Model (Stage 2)

Now combine the pretrained spectral embedding with scalar features for confidence scoring.

The fusion model sees:
- `ref_embed` (64-dim): pretrained spectral representation of the reference spectrum
- 12 scalar features: entropy_sim, sim_gap, delta_rt, delta_mda, adducts, etc.

Total input: 76-dim → MLP → P(correct).
The encoder is fine-tuned end-to-end during this stage.

In [ ]:
class ContrastiveFusionModel(nn.Module):
    """
    Stage 2: combine pretrained spectral encoder with scalar evidence.

    Architecture:
        ref_spectrum → [SpectralEncoder (pretrained)] → ref_embed (64)
        scalar_features (12) → [standardized]
        concat(ref_embed, scalar_features) → [Fusion MLP] → logit
    """

    def __init__(self, encoder, n_scalar_features=12, hidden=64, dropout=0.3):
        super().__init__()
        self.encoder = encoder  # pretrained, will be fine-tuned
        fusion_input = EMBED_DIM + n_scalar_features

        self.fusion = nn.Sequential(
            nn.Linear(fusion_input, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, ref_spectra, scalar_features):
        ref_emb = self.encoder(ref_spectra)
        fused = torch.cat([ref_emb, scalar_features], dim=-1)
        return self.fusion(fused).squeeze(-1)


def train_fusion(model, ref_tr, scalar_tr, y_tr, ref_val, scalar_val, y_val,
                 lr=5e-4, weight_decay=1e-4, epochs=150, batch_size=256, patience=20):
    """Train fusion model with early stopping."""
    n_pos = y_tr.sum()
    n_neg = len(y_tr) - n_pos
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # Lower LR for encoder (pretrained), higher for fusion head (new)
    optimizer = optim.Adam([
        {'params': model.encoder.parameters(), 'lr': lr * 0.1},  # fine-tune slowly
        {'params': model.fusion.parameters(), 'lr': lr},
    ], weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=7, factor=0.5)

    ref_t = torch.tensor(ref_tr, dtype=torch.float32)
    sc_t = torch.tensor(scalar_tr, dtype=torch.float32)
    y_t = torch.tensor(y_tr, dtype=torch.float32)
    ref_v = torch.tensor(ref_val, dtype=torch.float32)
    sc_v = torch.tensor(scalar_val, dtype=torch.float32)

    dataset = TensorDataset(ref_t, sc_t, y_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    best_auc = 0; best_state = None; wait = 0
    for epoch in range(epochs):
        model.train()
        for rb, sb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(rb, sb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(ref_v, sc_v).numpy()
            val_probs = 1.0 / (1.0 + np.exp(-val_logits))
            val_auc = roc_auc_score(y_val, val_probs)
        scheduler.step(-val_auc)

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return best_auc, epoch + 1


print(f'ContrastiveFusionModel: {sum(p.numel() for p in ContrastiveFusionModel(SpectralEncoder()).parameters()):,} params')

## Step 6 — Head-to-head evaluation

Same 5-fold GroupKFold as all previous experiments. Four models:
1. **Bayesian 3ch** (baseline, AUC 0.841)
2. **GBM 12 features** (best tabular model, AUC 0.882)
3. **Contrastive fusion** (pretrained encoder + 12 scalar features)
4. **Contrastive fusion (frozen)** — encoder NOT fine-tuned, to measure pretraining value alone

In [ ]:
ALL_COLS = [
    'entropy_similarity', 'sim_gap', 'signed_delta_rt', 'delta_mda',
    'spectral_entropy', 'hit_is_isf', 'hit_isf_no_ok',
    'compound_has_ok_adduct', 'n_candidate_adducts', 'n_candidates',
    'hit_is_dubious', 'polarity',
]
medians = top1[ALL_COLS].median()
X_scalar = top1[ALL_COLS].fillna(medians).values.astype(np.float32)

# CV groups
groups = top1['anno_ik14'].fillna('').values.copy()
for i in range(len(groups)):
    if groups[i] == '':
        groups[i] = '__no_ik14_%d' % i

gkf = GroupKFold(n_splits=5)

# Use only spectra that have reference peaks (95.5%)
# For the ~4.5% without, we'll fill ref_binned with zeros (the encoder will produce a generic embedding)

oof = {
    'gbm':            np.full(len(labels), np.nan),
    'contrastive_ft': np.full(len(labels), np.nan),  # fine-tuned encoder
    'contrastive_fz': np.full(len(labels), np.nan),  # frozen encoder
}
fold_aucs = {k: [] for k in oof}

# Save pretrained encoder state for resetting each fold
pretrained_state = {k: v.clone() for k, v in encoder.state_dict().items()}

t0 = time.time()
for fold, (train_idx, test_idx) in enumerate(gkf.split(X_scalar, labels, groups)):
    t_fold = time.time()

    # Scalar features
    scaler = StandardScaler()
    sc_tr = scaler.fit_transform(X_scalar[train_idx]).astype(np.float32)
    sc_te = scaler.transform(X_scalar[test_idx]).astype(np.float32)

    # Reference spectra (binned)
    ref_tr = ref_binned[train_idx]
    ref_te = ref_binned[test_idx]

    y_tr = labels[train_idx]
    y_te = labels[test_idx]

    # Nested val split
    n_val = int(len(sc_tr) * 0.2)
    sc_nn_tr, sc_nn_val = sc_tr[:-n_val], sc_tr[-n_val:]
    ref_nn_tr, ref_nn_val = ref_tr[:-n_val], ref_tr[-n_val:]
    y_nn_tr, y_nn_val = y_tr[:-n_val], y_tr[-n_val:]

    # ── GBM baseline ──
    gbm = GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=42
    )
    gbm.fit(sc_tr, y_tr)
    oof['gbm'][test_idx] = gbm.predict_proba(sc_te)[:, 1]

    # ── Contrastive fusion (fine-tuned) ──
    torch.manual_seed(42 + fold)
    enc_ft = SpectralEncoder()
    enc_ft.load_state_dict(pretrained_state)  # start from pretrained
    model_ft = ContrastiveFusionModel(enc_ft, n_scalar_features=len(ALL_COLS))
    _, ep_ft = train_fusion(model_ft, ref_nn_tr, sc_nn_tr, y_nn_tr,
                            ref_nn_val, sc_nn_val, y_nn_val)
    model_ft.eval()
    with torch.no_grad():
        logits = model_ft(torch.tensor(ref_te), torch.tensor(sc_te)).numpy()
        oof['contrastive_ft'][test_idx] = 1.0 / (1.0 + np.exp(-logits))

    # ── Contrastive fusion (frozen encoder) ──
    torch.manual_seed(42 + fold)
    enc_fz = SpectralEncoder()
    enc_fz.load_state_dict(pretrained_state)
    for p in enc_fz.parameters():
        p.requires_grad = False  # freeze encoder
    model_fz = ContrastiveFusionModel(enc_fz, n_scalar_features=len(ALL_COLS))
    _, ep_fz = train_fusion(model_fz, ref_nn_tr, sc_nn_tr, y_nn_tr,
                            ref_nn_val, sc_nn_val, y_nn_val)
    model_fz.eval()
    with torch.no_grad():
        logits = model_fz(torch.tensor(ref_te), torch.tensor(sc_te)).numpy()
        oof['contrastive_fz'][test_idx] = 1.0 / (1.0 + np.exp(-logits))

    for name in oof:
        fold_aucs[name].append(roc_auc_score(y_te, oof[name][test_idx]))

    elapsed = time.time() - t_fold
    print('Fold %d: GBM=%.4f  Contr(ft)=%.4f(%dep)  Contr(fz)=%.4f(%dep)  [%.1fs]' % (
        fold, fold_aucs['gbm'][-1],
        fold_aucs['contrastive_ft'][-1], ep_ft,
        fold_aucs['contrastive_fz'][-1], ep_fz,
        elapsed))

total_time = time.time() - t0

# ── Summary ──
print('\n' + '='*70)
print('%-25s  %8s  %9s  %9s' % ('Model', 'OOF AUC', 'Fold min', 'Fold max'))
print('-'*70)
for name, label in [('gbm', 'GBM (12 scalar)'),
                    ('contrastive_fz', 'Contrastive (frozen enc)'),
                    ('contrastive_ft', 'Contrastive (fine-tuned)')]:
    valid = ~np.isnan(oof[name])
    auc = roc_auc_score(labels[valid], oof[name][valid])
    print('%-25s  %8.4f  %9.4f  %9.4f' % (label, auc, min(fold_aucs[name]), max(fold_aucs[name])))
print('%-25s  %8s  %9s' % ('Bayesian 3ch', '0.8414', '(reference)'))
print('='*70)

# FDR table
print('\n%-25s  %8s  %6s  %8s  %6s' % ('Model', 'FDR@0.9', 'n', 'FDR@0.8', 'n'))
print('-' * 60)
for name, label in [('gbm', 'GBM (12 scalar)'),
                    ('contrastive_fz', 'Contrastive (frozen)'),
                    ('contrastive_ft', 'Contrastive (fine-tuned)')]:
    valid = ~np.isnan(oof[name])
    lab_v = labels[valid]
    pred_v = oof[name][valid]
    parts = ['%-25s' % label]
    for t in [0.9, 0.8]:
        called = pred_v >= t
        n = called.sum()
        n_fp = ((lab_v == 0) & called).sum()
        fdr = n_fp / n if n > 0 else 0
        parts.append('%7.1f%%  %6d' % (fdr*100, n))
    print('  '.join(parts))

print('\nTotal time (Stage 2): %.1fs' % total_time)

## Step 7 — What did the encoder learn? Ablation and embedding analysis

Key questions:
1. **Does the spectral embedding add value over scalar features alone?** (contrastive vs GBM)
2. **Does fine-tuning help?** (frozen vs fine-tuned encoder)
3. **What does the embedding capture that entropy_similarity doesn't?**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = {'gbm': 'tab:green', 'contrastive_fz': 'tab:blue', 'contrastive_ft': 'tab:red'}
display = {'gbm': 'GBM (12 scalar)', 'contrastive_fz': 'Contrastive (frozen)',
           'contrastive_ft': 'Contrastive (fine-tuned)'}

# ── Panel 1: ROC curves ──
ax = axes[0]
for name in oof:
    valid = ~np.isnan(oof[name])
    auc = roc_auc_score(labels[valid], oof[name][valid])
    fpr, tpr, _ = roc_curve(labels[valid], oof[name][valid])
    ax.plot(fpr, tpr, color=colors[name], lw=2, label='%s (%.3f)' % (display[name], auc))
ax.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.3)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC comparison')
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)

# ── Panel 2: Score distributions (contrastive fine-tuned) ──
ax = axes[1]
v = ~np.isnan(oof['contrastive_ft'])
ax.hist(oof['contrastive_ft'][v & (labels==1)], bins=50, density=True, alpha=0.5,
        color='steelblue', label='TP')
ax.hist(oof['contrastive_ft'][v & (labels==0)], bins=50, density=True, alpha=0.5,
        color='salmon', label='FP')
ax.set_xlabel('Contrastive fusion score')
ax.set_ylabel('Density')
ax.set_title('Score distribution (fine-tuned)')
ax.legend()

# ── Panel 3: Contrastive vs GBM scatter ──
ax = axes[2]
v = ~np.isnan(oof['contrastive_ft']) & ~np.isnan(oof['gbm'])
tp = labels[v] == 1
ax.scatter(oof['gbm'][v][tp], oof['contrastive_ft'][v][tp],
           s=5, alpha=0.3, color='steelblue', label='TP')
ax.scatter(oof['gbm'][v][~tp], oof['contrastive_ft'][v][~tp],
           s=5, alpha=0.3, color='salmon', label='FP')
ax.plot([0,1],[0,1], 'k--', lw=0.8)
ax.set_xlabel('GBM score'); ax.set_ylabel('Contrastive score')
ax.set_title('GBM vs Contrastive — where do they disagree?')
ax.legend()

plt.tight_layout()
plt.show()

# ── Embedding analysis: does the ref embedding add info beyond entropy_similarity? ──
# Compute correlation between ref embedding norm and entropy_similarity
encoder.eval()
with torch.no_grad():
    ref_emb_all = encoder(torch.tensor(ref_binned)).numpy()

# L2 norm of embedding (higher = more "confident" spectrum?)
emb_norm = np.linalg.norm(ref_emb_all, axis=1)  # should be ~1.0 since we normalize

# Cosine similarity between ref embedding and mean TP embedding vs mean FP embedding
tp_centroid = ref_emb_all[labels == 1].mean(axis=0)
tp_centroid /= np.linalg.norm(tp_centroid)
sim_to_tp = ref_emb_all @ tp_centroid

# Is this correlated with entropy_similarity?
from scipy.stats import spearmanr
esim = top1['entropy_similarity'].values
rho, pval = spearmanr(sim_to_tp[ref_available], esim[ref_available])
print(f'Correlation between embedding-TP-similarity and entropy_similarity: rho={rho:.3f} (p={pval:.1e})')

# AUC of embedding-based similarity alone
emb_auc = roc_auc_score(labels[ref_available], sim_to_tp[ref_available])
esim_auc = roc_auc_score(labels[ref_available], esim[ref_available])
print(f'AUC of embedding sim-to-TP-centroid: {emb_auc:.3f}')
print(f'AUC of entropy_similarity:           {esim_auc:.3f}')
print(f'If embedding AUC > entropy_sim AUC, the encoder learned something beyond simple spectral similarity')